# Week 5: Community Detection — Assignment

**Learning objectives** — In this assignment you will:

- Implement modularity from scratch using the null-model formula
- Build the Girvan-Newman edge-removal algorithm from scratch
- Compute Normalized Mutual Information (NMI) between partitions from scratch
- Sweep the Louvain resolution parameter and track quality metrics
- Quantify the stability of a stochastic community detection algorithm

## Grading

| Section | Function | Points |
|---------|----------|--------|
| 1 | `compute_modularity(G, partition)` | 20 |
| 2 | `girvan_newman(G, k)` | 25 |
| 3 | `nmi_score(partition_a, partition_b)` | 15 |
| 4 | `resolution_sweep(G, resolutions)` | 10 |
| 5 | `community_stability(G, n_runs)` | 15 |
| — | Written Questions | 15 |
| | **Total** | **100** |

## Before You Start

This assignment builds on the Week 5 lab. Make sure you are comfortable with:

- **Community definition** — dense connections within, sparse connections between groups (Lab Section 2)
- **Modularity (Q)** — compares actual edge density within communities to a random null model; Q ≈ 0 is random, Q > 0.3 is meaningful (Lab Section 3)
- **Louvain algorithm** — greedy modularity maximization with local moves + aggregation (Lab Section 4)
- **Girvan-Newman** — top-down approach: iteratively remove highest-betweenness edges to split the graph (Lab Section 5)
- **NMI** — Normalized Mutual Information measures agreement between two partitions; 1.0 = perfect match, 0.0 = independent (Lab Section 7)
- **Resolution parameter** — controls community granularity; higher resolution → more, smaller communities (Lab Section 10)

### Implementation constraints

The following functions are **banned** from your solutions:

| Banned function | Sections |
|----------------|----------|
| `nx.community.modularity` | 1, 4 |
| `nx.community.girvan_newman` | 2 |
| `sklearn.metrics.normalized_mutual_info_score`, `sklearn.metrics.mutual_info_score` | 3, 5 |

You **may** use: `G.neighbors()`, `G.nodes()`, `G.edges()`, `G.degree()`, `G.number_of_edges()`, `G.number_of_nodes()`, `nx.Graph()`, `nx.edge_betweenness_centrality()`, `nx.connected_components()`, `nx.community.louvain_communities()`, `nx.average_clustering()`, `np.log()`, `collections.Counter`.

**Important**: Later sections build on earlier ones. You must **reuse your own implementations**:
- Section 4 must use your `compute_modularity` from Section 1
- Section 5 must use your `nmi_score` from Section 3

In [3]:
import networkx as nx
import numpy as np
from netsci.loaders import load_graph
from netsci.utils import SEED

In [4]:
G_karate = load_graph("karate")
G_football = load_graph("football")

# Ground truth for karate (2 factions)
gt_karate = [
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Mr. Hi"},
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Officer"},
]

# Ground truth for football (12 conferences)
_conf_map = {}
for n in G_football.nodes():
    c = G_football.nodes[n]["conference"]
    _conf_map.setdefault(c, set()).add(n)
gt_football = list(_conf_map.values())

karate: 34 nodes, 78 edges (undirected)
football: 115 nodes, 613 edges (undirected)


---
## Section 1: Modularity from Scratch (20 pts)

Implement the **unweighted** modularity formula:

$$Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where:
- $A_{ij} \in \{0, 1\}$ is the unweighted adjacency matrix entry (ignore edge weights)
- $k_i$ is the degree of node $i$
- $m$ is the total number of edges
- $\delta(c_i, c_j) = 1$ if nodes $i$ and $j$ are in the same community

**Equivalent per-community form** (easier to implement):

$$Q = \sum_{c} \left[ \frac{L_c}{m} - \left( \frac{d_c}{2m} \right)^2 \right]$$

where $L_c$ is the number of internal edges in community $c$ and $d_c = \sum_{i \in c} k_i$.

**Do not use** `nx.community.modularity`.

In [15]:
def compute_modularity(G, partition):
    """Compute unweighted modularity of a partition from scratch.

    Parameters
    ----------
    G : nx.Graph
    partition : list of sets
        Each set contains the nodes in one community.

    Returns
    -------
    float
    """
    m = G.number_of_edges()
    if m == 0:
        return 0.0

    two_m = 2 * m
    degree = dict(G.degree())

    q = 0.0
    for community in partition:
        # L_c: internal edges in this community
        subgraph = G.subgraph(community)
        l_c = subgraph.number_of_edges()

        # d_c: sum of degrees for nodes in this community
        d_c = sum(degree[n] for n in community)

        q += (l_c / m) - (d_c / two_m) ** 2

    return float(q)

In [16]:
# --- Validation ---
# Note: we use weight=None because the formula above is unweighted
_Q = compute_modularity(G_karate, gt_karate)
_Q_nx = nx.community.modularity(G_karate, gt_karate, weight=None)
assert abs(_Q - _Q_nx) < 1e-6, f"Got {_Q}, expected {_Q_nx}"
print(f"Karate GT modularity: {_Q:.6f} (expected: {_Q_nx:.6f})")

# Test with Louvain output on football
_louv = list(nx.community.louvain_communities(G_football, seed=SEED))
_Q2 = compute_modularity(G_football, _louv)
_Q2_nx = nx.community.modularity(G_football, _louv, weight=None)
assert abs(_Q2 - _Q2_nx) < 1e-6, f"Football: got {_Q2}, expected {_Q2_nx}"
print(f"Football Louvain modularity: {_Q2:.6f}")

# Test edge case: singleton partition (each node is its own community)
_single = [{n} for n in G_karate.nodes()]
_Q3 = compute_modularity(G_karate, _single)
_Q3_nx = nx.community.modularity(G_karate, _single, weight=None)
assert abs(_Q3 - _Q3_nx) < 1e-6, f"Singleton: got {_Q3}, expected {_Q3_nx}"
print(f"Singleton partition modularity: {_Q3:.6f} (should be negative)")

# Test: one giant community (all nodes together)
_all_one = [set(G_karate.nodes())]
_Q4 = compute_modularity(G_karate, _all_one)
assert abs(_Q4 - 0.0) < 1e-6, f"Single community should give Q=0, got {_Q4}"
print(f"Single community modularity: {_Q4:.6f} (should be 0)")
print("Section 1 passed!")

Karate GT modularity: 0.358235 (expected: 0.358235)
Football Louvain modularity: 0.604570
Singleton partition modularity: -0.049803 (should be negative)
Single community modularity: 0.000000 (should be 0)
Section 1 passed!


---
## Section 2: Girvan-Newman from Scratch (25 pts)

Implement the Girvan-Newman algorithm **from scratch**. **Do NOT** call `nx.community.girvan_newman`.

The algorithm:

1. **Start** with a copy of the input graph
2. **Repeat** until the graph has at least `k` connected components:
   - Compute edge betweenness centrality for all edges (use `nx.edge_betweenness_centrality`)
   - Remove the edge with the **highest** betweenness centrality
   - If multiple edges tie for highest betweenness, remove any one of them
3. **Return** the connected components as a list of sets

**Implementation details:**
- Work on a **copy** of the graph — do not modify the original
- After removing edges, use `nx.connected_components()` to count components
- Return the partition as a list of sets (each set = one community), sorted by size descending

In [17]:
def girvan_newman(G, k=2):
    """Split a graph into k communities using Girvan-Newman.

    Do NOT call nx.community.girvan_newman.

    Parameters
    ----------
    G : nx.Graph
        Must be connected.
    k : int
        Target number of communities.

    Returns
    -------
    list of sets — partition into k communities, sorted by size descending
    """
    if k <= 1:
        return [set(G.nodes())]

    work = G.copy()

    while nx.number_connected_components(work) < k and work.number_of_edges() > 0:
        bet = nx.edge_betweenness_centrality(work)
        edge_to_remove = max(bet, key=bet.get)
        work.remove_edge(*edge_to_remove)

    parts = [set(c) for c in nx.connected_components(work)]
    parts.sort(key=len, reverse=True)
    return parts

In [18]:
# --- Validation ---
# Karate club: split into 2
_gn2 = girvan_newman(G_karate, k=2)
assert isinstance(_gn2, list)
assert len(_gn2) == 2, f"Expected 2 communities, got {len(_gn2)}"
assert all(isinstance(c, set) for c in _gn2)
_all_nodes_gn = set()
for c in _gn2:
    _all_nodes_gn |= c
assert _all_nodes_gn == set(G_karate.nodes()), "All nodes must be assigned"

# Should produce a decent partition
_Q_gn = nx.community.modularity(G_karate, _gn2, weight=None)
assert _Q_gn > 0.3, f"GN modularity {_Q_gn:.3f} too low for karate (expected > 0.3)"
print(f"GN(k=2) on karate: sizes={sorted([len(c) for c in _gn2], reverse=True)}, Q={_Q_gn:.4f}")

# Sorted by size descending
assert len(_gn2[0]) >= len(_gn2[1]), "Communities should be sorted by size descending"

# Original graph unchanged
assert G_karate.number_of_edges() == 78, "Original graph must not be modified"

# Karate club: split into 4
_gn4 = girvan_newman(G_karate, k=4)
assert len(_gn4) == 4, f"Expected 4 communities, got {len(_gn4)}"
_all4 = set()
for c in _gn4:
    _all4 |= c
assert _all4 == set(G_karate.nodes())
_Q_gn4 = nx.community.modularity(G_karate, _gn4, weight=None)
print(f"GN(k=4) on karate: sizes={sorted([len(c) for c in _gn4], reverse=True)}, Q={_Q_gn4:.4f}")

# Football: split into 12 conferences
_gn_fb = girvan_newman(G_football, k=12)
assert len(_gn_fb) == 12, f"Expected 12 communities, got {len(_gn_fb)}"
_Q_gn_fb = nx.community.modularity(G_football, _gn_fb, weight=None)
assert _Q_gn_fb > 0.4, f"GN modularity on football {_Q_gn_fb:.3f} too low"
print(f"GN(k=12) on football: Q={_Q_gn_fb:.4f}")
print("Section 2 passed!")

GN(k=2) on karate: sizes=[19, 15], Q=0.3600
GN(k=4) on karate: sizes=[18, 10, 5, 1], Q=0.3632
GN(k=12) on football: Q=0.5973
Section 2 passed!


---
## Section 3: Normalized Mutual Information (15 pts)

Implement NMI from scratch. **Do NOT** use `sklearn.metrics.normalized_mutual_info_score` or any library NMI function.

NMI measures the agreement between two partitions on a scale of 0 (independent) to 1 (identical).

$$\text{NMI}(U, V) = \frac{2 \cdot I(U; V)}{H(U) + H(V)}$$

where:
- $H(U) = -\sum_i \frac{|U_i|}{N} \ln \frac{|U_i|}{N}$ is the entropy of partition $U$
- $I(U; V) = \sum_i \sum_j \frac{|U_i \cap V_j|}{N} \ln \frac{N \cdot |U_i \cap V_j|}{|U_i| \cdot |V_j|}$ is the mutual information
- Skip terms where $|U_i \cap V_j| = 0$ (since $0 \cdot \ln 0 = 0$)
- If $H(U) + H(V) = 0$ (both partitions place all nodes in one group), return 1.0

In [19]:
def nmi_score(partition_a, partition_b):
    """Compute Normalized Mutual Information between two partitions.

    Do NOT use sklearn.metrics.normalized_mutual_info_score or any
    library NMI function.

    Parameters
    ----------
    partition_a : list of sets
    partition_b : list of sets

    Returns
    -------
    float (between 0 and 1)
    """
    nodes_a = set().union(*partition_a) if partition_a else set()
    nodes_b = set().union(*partition_b) if partition_b else set()
    if nodes_a != nodes_b:
        raise ValueError("Partitions must cover the same set of nodes.")

    n = len(nodes_a)
    if n == 0:
        return 1.0

    def entropy(partition):
        h = 0.0
        for community in partition:
            p = len(community) / n
            if p > 0:
                h -= p * np.log(p)
        return h

    h_a = entropy(partition_a)
    h_b = entropy(partition_b)

    mi = 0.0
    for a in partition_a:
        size_a = len(a)
        if size_a == 0:
            continue
        for b in partition_b:
            inter = len(a & b)
            if inter == 0:
                continue
            size_b = len(b)
            p_ab = inter / n
            mi += p_ab * np.log((n * inter) / (size_a * size_b))

    denom = h_a + h_b
    if denom == 0:
        return 1.0

    nmi = (2.0 * mi) / denom
    # Numerical safety for tiny floating errors.
    return float(min(1.0, max(0.0, nmi)))

In [20]:
# --- Validation ---
from sklearn.metrics import normalized_mutual_info_score as _sklearn_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

# Perfect match should be 1.0
_nmi_perfect = nmi_score(gt_karate, gt_karate)
assert abs(_nmi_perfect - 1.0) < 1e-6, f"Perfect NMI should be 1.0, got {_nmi_perfect}"
print(f"NMI(GT, GT) = {_nmi_perfect:.4f}")

# All-in-one vs all-in-one should be 1.0 (edge case)
_all_one_a = [set(G_karate.nodes())]
_all_one_b = [set(G_karate.nodes())]
_nmi_trivial = nmi_score(_all_one_a, _all_one_b)
assert abs(_nmi_trivial - 1.0) < 1e-6, f"Trivial NMI should be 1.0, got {_nmi_trivial}"

# Louvain vs ground truth on karate
_louv_k = list(nx.community.louvain_communities(G_karate, seed=SEED))
_nmi_louv = nmi_score(_louv_k, gt_karate)
_nodes_k = list(G_karate.nodes())
_nmi_sklearn = _sklearn_nmi(
    _to_labels(_louv_k, _nodes_k), _to_labels(gt_karate, _nodes_k)
)
assert abs(_nmi_louv - _nmi_sklearn) < 0.05, (
    f"NMI(Louvain, GT) = {_nmi_louv:.4f}, sklearn says {_nmi_sklearn:.4f}"
)
print(f"NMI(Louvain, GT) on karate = {_nmi_louv:.4f} (sklearn: {_nmi_sklearn:.4f})")

# Football: Louvain vs conference ground truth
_louv_fb = list(nx.community.louvain_communities(G_football, seed=SEED))
_nmi_fb = nmi_score(_louv_fb, gt_football)
_nodes_fb = list(G_football.nodes())
_nmi_fb_sk = _sklearn_nmi(
    _to_labels(_louv_fb, _nodes_fb), _to_labels(gt_football, _nodes_fb)
)
assert abs(_nmi_fb - _nmi_fb_sk) < 0.05, (
    f"Football NMI = {_nmi_fb:.4f}, sklearn = {_nmi_fb_sk:.4f}"
)
print(f"NMI(Louvain, GT) on football = {_nmi_fb:.4f}")

# NMI should be > 0 but < 1 for imperfect match
assert 0 < _nmi_louv < 1.0, f"Expected 0 < NMI < 1, got {_nmi_louv}"
print("Section 3 passed!")

NMI(GT, GT) = 1.0000
NMI(Louvain, GT) on karate = 0.5942 (sklearn: 0.5942)
NMI(Louvain, GT) on football = 0.8903
Section 3 passed!


---
## Section 4: Resolution Sweep (10 pts)

Sweep over a list of Louvain resolution values and collect quality metrics at each resolution.

**Implementation requirements:**
- **Reuse your own** `compute_modularity` from Section 1 (do NOT call `nx.community.modularity`)
- Use `nx.community.louvain_communities(G, resolution=r, seed=SEED)` for detection
- Return a dictionary with keys:
  - `"resolution"` — list of resolution values
  - `"modularity"` — list of modularity values (from YOUR `compute_modularity`)
  - `"n_communities"` — list of community counts
  - `"best_resolution"` — the resolution that achieved the highest modularity

In [21]:
def resolution_sweep(G, resolutions):
    """Sweep Louvain resolution and collect quality metrics.

    Reuse your compute_modularity() from Section 1.
    Do NOT call nx.community.modularity.

    Parameters
    ----------
    G : nx.Graph
    resolutions : list of float

    Returns
    -------
    dict with keys 'resolution', 'modularity', 'n_communities', 'best_resolution'
    """
    q_values = []
    n_communities = []

    for r in resolutions:
        part = list(nx.community.louvain_communities(G, resolution=r, seed=SEED))
        q_values.append(compute_modularity(G, part))
        n_communities.append(len(part))

    best_idx = q_values.index(max(q_values)) if q_values else None
    best_resolution = resolutions[best_idx] if best_idx is not None else None

    return {
        "resolution": list(resolutions),
        "modularity": q_values,
        "n_communities": n_communities,
        "best_resolution": best_resolution,
    }

In [22]:
# --- Validation ---
_res = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
_result = resolution_sweep(G_football, _res)

assert set(_result.keys()) == {"resolution", "modularity", "n_communities", "best_resolution"}
assert len(_result["resolution"]) == len(_res)
assert len(_result["modularity"]) == len(_res)
assert len(_result["n_communities"]) == len(_res)

# More communities at higher resolution
assert _result["n_communities"][-1] > _result["n_communities"][0], (
    "Higher resolution should produce more communities"
)

# Verify modularity values against nx (unweighted)
for i, r in enumerate(_res):
    _p = list(nx.community.louvain_communities(G_football, resolution=r, seed=SEED))
    _q_expected = nx.community.modularity(G_football, _p, weight=None)
    assert abs(_result["modularity"][i] - _q_expected) < 1e-6, (
        f"Modularity mismatch at r={r}: got {_result['modularity'][i]}, expected {_q_expected}"
    )

# best_resolution should be in the input list
assert _result["best_resolution"] in _res, (
    f"best_resolution {_result['best_resolution']} not in input resolutions"
)

# best_resolution should correspond to max modularity
_best_idx = _result["modularity"].index(max(_result["modularity"]))
assert abs(_result["best_resolution"] - _res[_best_idx]) < 1e-9, (
    f"best_resolution should match the resolution with highest Q"
)

print("Resolution | Communities | Modularity")
print("-" * 42)
for i in range(len(_res)):
    marker = " <-- best" if abs(_res[i] - _result["best_resolution"]) < 1e-9 else ""
    print(f"    {_res[i]:.1f}    |     {_result['n_communities'][i]:2d}      | {_result['modularity'][i]:.4f}{marker}")
print(f"\nBest resolution: {_result['best_resolution']}")
print("Section 4 passed!")

Resolution | Communities | Modularity
------------------------------------------
    0.5    |      6      | 0.5828
    0.8    |      7      | 0.6006
    1.0    |     10      | 0.6046 <-- best
    1.2    |     10      | 0.6044
    1.5    |     12      | 0.6005
    2.0    |     12      | 0.6005
    3.0    |     12      | 0.6005

Best resolution: 1.0
Section 4 passed!


---
## Section 5: Community Stability (15 pts)

Louvain is stochastic — different random seeds produce different partitions.
Measure how stable the algorithm is by running it multiple times and computing the
average pairwise NMI between all pairs of resulting partitions.

**Implementation requirements:**
- Run `nx.community.louvain_communities(G, seed=i)` for `i = 0, 1, ..., n_runs - 1`
- **Reuse your own** `nmi_score` from Section 3 (do NOT use sklearn)
- Compute NMI for every pair `(i, j)` where `i < j`
- Return a dictionary with:
  - `"mean_nmi"` — average pairwise NMI (float)
  - `"min_nmi"` — minimum pairwise NMI (float)
  - `"max_nmi"` — maximum pairwise NMI (float)
  - `"n_unique"` — number of distinct community counts observed across runs (int)

In [23]:
def community_stability(G, n_runs=10):
    """Measure stability of Louvain across multiple runs.

    Reuse your nmi_score() from Section 3.
    Do NOT use sklearn.metrics.normalized_mutual_info_score.

    Parameters
    ----------
    G : nx.Graph
    n_runs : int

    Returns
    -------
    dict with keys 'mean_nmi', 'min_nmi', 'max_nmi', 'n_unique'
    """
    if n_runs <= 0:
        raise ValueError("n_runs must be >= 1")

    partitions = [list(nx.community.louvain_communities(G, seed=i)) for i in range(n_runs)]
    unique_counts = {len(p) for p in partitions}

    nmi_values = []
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            nmi_values.append(nmi_score(partitions[i], partitions[j]))

    if not nmi_values:
        # For n_runs == 1 there are no pairs to compare.
        return {
            "mean_nmi": 1.0,
            "min_nmi": 1.0,
            "max_nmi": 1.0,
            "n_unique": len(unique_counts),
        }

    return {
        "mean_nmi": float(np.mean(nmi_values)),
        "min_nmi": float(np.min(nmi_values)),
        "max_nmi": float(np.max(nmi_values)),
        "n_unique": len(unique_counts),
    }

In [24]:
# --- Validation ---
_stab_k = community_stability(G_karate, n_runs=10)
assert set(_stab_k.keys()) == {"mean_nmi", "min_nmi", "max_nmi", "n_unique"}
assert 0 <= _stab_k["min_nmi"] <= _stab_k["mean_nmi"] <= _stab_k["max_nmi"] <= 1.0
assert isinstance(_stab_k["n_unique"], int) and _stab_k["n_unique"] >= 1

print(f"Karate stability (10 runs):")
print(f"  Mean NMI: {_stab_k['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_k['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_k['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_k['n_unique']}")

_stab_fb = community_stability(G_football, n_runs=10)
assert 0 <= _stab_fb["min_nmi"] <= _stab_fb["mean_nmi"] <= _stab_fb["max_nmi"] <= 1.0

# Football should be reasonably stable (clear community structure)
assert _stab_fb["mean_nmi"] > 0.85, (
    f"Football mean NMI {_stab_fb['mean_nmi']:.3f} too low — expected > 0.85"
)
print(f"\nFootball stability (10 runs):")
print(f"  Mean NMI: {_stab_fb['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_fb['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_fb['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_fb['n_unique']}")

# Verify against sklearn
from sklearn.metrics import normalized_mutual_info_score as _sk_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

_parts = [list(nx.community.louvain_communities(G_football, seed=i)) for i in range(10)]
_nodes_fb = list(G_football.nodes())
_sk_nmis = []
for i in range(10):
    for j in range(i + 1, 10):
        _sk_nmis.append(_sk_nmi(
            _to_labels(_parts[i], _nodes_fb),
            _to_labels(_parts[j], _nodes_fb)
        ))
_sk_mean = np.mean(_sk_nmis)
assert abs(_stab_fb["mean_nmi"] - _sk_mean) < 0.05, (
    f"Mean NMI {_stab_fb['mean_nmi']:.4f} differs from sklearn {_sk_mean:.4f}"
)
print(f"  (sklearn verification: mean={_sk_mean:.4f})")
print("Section 5 passed!")

Karate stability (10 runs):
  Mean NMI: 0.9214
  Min NMI:  0.6880
  Max NMI:  1.0000
  Unique community counts: 2

Football stability (10 runs):
  Mean NMI: 0.9629
  Min NMI:  0.9198
  Max NMI:  1.0000
  Unique community counts: 2
  (sklearn verification: mean=0.9629)
Section 5 passed!


---
## Written Questions (15 pts)

### Question 1 (5 pts)

Run Girvan-Newman and Louvain on the karate club with `k=2` / default resolution and compare:

(a) Report the community sizes and modularity Q for each algorithm. Are the partitions identical? If not, which nodes are assigned differently?

(b) Girvan-Newman is $O(m^2 n)$ while Louvain is nearly $O(n \log n)$. Given their quality and speed trade-off, when would you choose Girvan-Newman over Louvain in practice?

(c) Compute the NMI between the two partitions (using your `nmi_score`). Is the disagreement large or small? What does this tell you about the "uniqueness" of community structure in this network?

**You must include numerical values from your code.** Show the code cell you used to compute them.

*Hints to guide your thinking:*
- *Girvan-Newman is deterministic (edge betweenness gives a unique ranking), while Louvain is stochastic. What are the implications?*
- *Think about "bridge" nodes between communities — are they the ones that differ?*
- *Consider the trade-off: Girvan-Newman gives a full dendrogram (hierarchical decomposition), while Louvain gives a flat partition at one level.*

In [28]:
# Q1 calculations: Girvan-Newman vs Louvain on karate

def _node_to_comm(partition):
    m = {}
    for i, comm in enumerate(partition):
        for node in comm:
            m[node] = i
    return m

_gn2_q1 = girvan_newman(G_karate, k=2)
_louv_q1 = list(nx.community.louvain_communities(G_karate, seed=SEED))

_q1_sizes_gn = sorted([len(c) for c in _gn2_q1], reverse=True)
_q1_sizes_louv = sorted([len(c) for c in _louv_q1], reverse=True)
_q1_q_gn = compute_modularity(G_karate, _gn2_q1)
_q1_q_louv = compute_modularity(G_karate, _louv_q1)
_q1_nmi = nmi_score(_gn2_q1, _louv_q1)

# Nodes in Louvain communities that mix both GN groups (boundary disagreements)
_map_gn = _node_to_comm(_gn2_q1)
_q1_disagreement_nodes = []
for comm in _louv_q1:
    gn_labels = {_map_gn[n] for n in comm}
    if len(gn_labels) > 1:
        _q1_disagreement_nodes.extend(sorted(comm))
_q1_disagreement_nodes = sorted(set(_q1_disagreement_nodes))

print("Q1 metrics")
print("GN sizes:", _q1_sizes_gn)
print("Louvain sizes:", _q1_sizes_louv)
print(f"Q_GN={_q1_q_gn:.6f}")
print(f"Q_Louvain={_q1_q_louv:.6f}")
print(f"NMI(GN, Louvain)={_q1_nmi:.6f}")
print("Nodes in mixed (disagreement) Louvain communities:", _q1_disagreement_nodes)

Q1 metrics
GN sizes: [19, 15]
Louvain sizes: [14, 10, 6, 4]
Q_GN=0.359961
Q_Louvain=0.390450
NMI(GN, Louvain)=0.616132
Nodes in mixed (disagreement) Louvain communities: [1, 2, 3, 7, 12, 13]


**Your Answer:**

(a) Am folosit celula de cod de mai sus (Q1 calculations). Rezultatele sunt:
- Girvan-Newman (`k=2`): mărimi comunități `[19, 15]`, modularitate $Q=0.359961$.
- Louvain (rezoluție implicită): mărimi comunități `[14, 10, 6, 4]`, modularitate $Q=0.390450$.

Partițiile **nu sunt identice** (nici măcar ca număr de comunități). Nodurile care apar în comunități Louvain „mixte” față de split-ul GN (deci la graniță între cele două perspective) sunt: `[1, 2, 3, 7, 12, 13]`.

(b) Aș alege Girvan-Newman când:
- vreau o descompunere ierarhică (dendrogramă), nu doar o partiție finală;
- graful este mic/mediu și interpretabilitatea „bridge edges” contează mai mult decât viteza;
- fac analiză explicativă (de ce se rupe rețeaua), nu procesare la scară mare.

Pentru grafuri mari sau workflow operațional, Louvain este de obicei preferat datorită costului mult mai mic.

(c) NMI între GN și Louvain este $0.616132$. Dezacordul este **moderat**: structura comunitară are un nucleu comun, dar există ambiguitate reală în zona nodurilor de frontieră. Asta sugerează că soluția comunitară nu este complet unică; mai multe partiții rezonabile pot coexista.

### Question 2 (5 pts)

Fortunato & Barthélemy (2007) proved that modularity optimization has a **resolution limit**: it may fail to detect communities smaller than $\sim\sqrt{2m}$ nodes, where $m$ is the total number of edges.

(a) For the football network, compute $\sqrt{2m}$. How many of the 12 ground-truth conferences have fewer members than this threshold?

(b) Yet Louvain still finds ~10 communities with Q > 0.55 on this network. Explain why the resolution limit doesn't completely destroy performance here. (*Hint: the worst case for the resolution limit involves specific graph topologies.*)

(c) **Use your Section 4 results**: At which resolution does the number of detected communities best match the 12 ground-truth conferences? Is this the same resolution that maximizes modularity? What does this discrepancy (if any) tell you about using modularity as the sole quality criterion?

*Hints to guide your thinking:*
- *The resolution limit is a worst-case result for two cliques connected by a single edge. Real communities have denser internal connections and sparser inter-community links.*
- *Compare the resolution that gives ~12 communities vs the one that maximizes Q. If they differ, it means maximizing Q doesn't recover the "right" number of communities.*
- *This is why the resolution parameter exists — it lets you override Q-maximization and zoom in/out.*

In [26]:
# Q2 calculations: resolution limit and resolution sweep interpretation
_m_fb = G_football.number_of_edges()
_q2_threshold = np.sqrt(2 * _m_fb)
_q2_conf_sizes = sorted([len(c) for c in gt_football])
_q2_below = sum(1 for s in _q2_conf_sizes if s < _q2_threshold)

_q2_res = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
_q2_sweep = resolution_sweep(G_football, _q2_res)
_q2_counts = _q2_sweep["n_communities"]
_q2_mod = _q2_sweep["modularity"]
_q2_best_q_res = _q2_sweep["best_resolution"]

# Resolution whose detected number of communities is closest to 12
_q2_target = 12
_q2_best_count_idx = min(range(len(_q2_res)), key=lambda i: (abs(_q2_counts[i] - _q2_target), i))
_q2_best_count_res = _q2_res[_q2_best_count_idx]

print("Q2 metrics")
print(f"m={_m_fb}")
print(f"sqrt(2m)={_q2_threshold:.4f}")
print("conference sizes:", _q2_conf_sizes)
print(f"#conferences with size < sqrt(2m): {_q2_below} / {len(_q2_conf_sizes)}")
print("resolutions:", _q2_res)
print("n_communities:", _q2_counts)
print("modularity:", [round(x, 6) for x in _q2_mod])
print(f"resolution with max Q: {_q2_best_q_res}")
print(f"resolution closest to 12 communities: {_q2_best_count_res}")

Q2 metrics
m=613
sqrt(2m)=35.0143
conference sizes: [5, 7, 8, 8, 9, 10, 10, 10, 11, 12, 12, 13]
#conferences with size < sqrt(2m): 12 / 12
resolutions: [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
n_communities: [6, 7, 10, 10, 12, 12, 12]
modularity: [0.58275, 0.600642, 0.60457, 0.604429, 0.600517, 0.600517, 0.600517]
resolution with max Q: 1.0
resolution closest to 12 communities: 1.5


**Your Answer:**

(a) Din celula de cod Q2:
- Pentru football, $m=613$.
- Pragul teoretic este $\sqrt{2m}=\sqrt{1226}=35.0143$.
- Dimensiunile conferințelor sunt `[5, 7, 8, 8, 9, 10, 10, 10, 11, 12, 12, 13]`.
- Număr conferințe sub prag: **12 din 12**.

(b) Deși toate comunitățile reale sunt sub pragul de worst-case, performanța nu se prăbușește fiindcă limita de rezoluție este un rezultat teoretic pentru topologii nefavorabile (ex. cliques legate foarte slab într-un anumit mod). În football, separarea între conferințe este încă suficient de clară structural (densitate internă relativ mare + legături inter-conferință mai rare), deci Louvain poate recupera o partiționare bună (Q > 0.55).

(c) Din Section 4 / Q2:
- Rezoluția care maximizează modularitatea: `1.0` (Q = `0.604570`, 10 comunități).
- Rezoluția care potrivește cel mai bine cele 12 conferințe: `1.5` (12 comunități, Q = `0.600517`).

Nu sunt aceleași. Diferența arată că maximizarea lui $Q$ ca unic criteriu poate favoriza o granularitate diferită de „adevărul” semantic al datelor; de aceea rezoluția trebuie calibrată cu obiectivul analizei (număr de grupuri, interpretabilitate, knowledge de domeniu).

### Question 3 (5 pts)

Your Section 5 measures the stability of Louvain across runs.

(a) Report the mean, min, and max pairwise NMI for both karate and football. Which network has more stable community assignments? Why? (*Think about the structure of each network.*)

(b) Identify the "unstable" nodes — nodes most likely to switch communities across runs. What structural property do these nodes share? (*Hint: think about their position relative to community boundaries.*)

(c) If you needed to produce a single "consensus" partition from 100 Louvain runs, describe an algorithm to do so. How would you use NMI or modularity to select or construct the best result?

*Hints to guide your thinking:*
- *Small networks have fewer constraints, so the algorithm has more "freedom" to assign ambiguous nodes differently.*
- *Bridge nodes, nodes with low clustering, or nodes at the boundary of communities are structurally ambiguous.*
- *For consensus: consider building a co-occurrence matrix (how often each pair of nodes ends up in the same community) and clustering that. Alternatively, pick the run whose partition has the highest average NMI with all other runs.*

In [27]:
# Q3 calculations: stability + unstable nodes over 100 runs

def _node_instability_scores(G, n_runs=100):
    from collections import Counter

    partitions = [list(nx.community.louvain_communities(G, seed=i)) for i in range(n_runs)]
    scores = {}

    for node in G.nodes():
        labels = []
        for part in partitions:
            comm = next(c for c in part if node in c)
            labels.append(frozenset(comm))
        counts = Counter(labels)
        max_freq = counts.most_common(1)[0][1]
        scores[node] = 1.0 - (max_freq / n_runs)

    return scores

_q3_karate = community_stability(G_karate, n_runs=10)
_q3_football = community_stability(G_football, n_runs=10)

_q3_scores_k = _node_instability_scores(G_karate, n_runs=100)
_q3_top_k = sorted(_q3_scores_k.items(), key=lambda x: x[1], reverse=True)[:8]

print("Q3 metrics")
print("karate:", _q3_karate)
print("football:", _q3_football)
print("Top unstable karate nodes (node, instability):")
for node, score in _q3_top_k:
    print(
        f"  node={node:2d} score={score:.3f} degree={G_karate.degree(node):2d} "
        f"clustering={nx.clustering(G_karate, node):.3f}"
    )

Q3 metrics
karate: {'mean_nmi': 0.9213835516434453, 'min_nmi': 0.688020958870953, 'max_nmi': 1.0, 'n_unique': 2}
football: {'mean_nmi': 0.9629220708074464, 'min_nmi': 0.9198244769280097, 'max_nmi': 1.0, 'n_unique': 2}
Top unstable karate nodes (node, instability):
  node= 8 score=0.550 degree= 5 clustering=0.500
  node= 9 score=0.550 degree= 2 clustering=0.000
  node=14 score=0.550 degree= 2 clustering=1.000
  node=15 score=0.550 degree= 2 clustering=1.000
  node=18 score=0.550 degree= 2 clustering=1.000
  node=20 score=0.550 degree= 2 clustering=1.000
  node=22 score=0.550 degree= 2 clustering=1.000
  node=23 score=0.550 degree= 5 clustering=0.400


**Your Answer:**

(a) Din celula Q3 (10 rulări):
- Karate: mean NMI = `0.9214`, min = `0.6880`, max = `1.0000`.
- Football: mean NMI = `0.9629`, min = `0.9198`, max = `1.0000`.

Network-ul mai stabil este **football** (mean/min mai mari). Intuitiv, are structură comunitară mai clară și mai puține noduri ambigue relativ la scară.

(b) Noduri instabile (karate, 100 rulări, scor instabilitate $=1-\frac{f_{max}}{100}$):
- top: `8, 9, 14, 15, 18, 20, 22, 23` (scor ~`0.55`).

Aceste noduri tind să fie la frontieră între comunități sau în poziții structural ambigue: legături de punte, grad mic și/sau conectivitate care permite apartenență plauzibilă la mai multe grupuri.

(c) O metodă de consens pentru 100 rulări:
1. Rulez Louvain de 100 ori și construiesc matricea de co-ocurență $C$, unde $C_{ij}$ = fracția rulărilor în care nodurile $i,j$ sunt în aceeași comunitate.
2. Praghez / clusterizez matricea $C$ (de exemplu pe un graf ponderat cu muchii pentru co-ocurență mare) pentru a obține partiția de consens.
3. Evaluez partiția de consens prin modularitate și prin NMI mediu față de cele 100 de rulări.
4. Alternativ simplu: aleg rularea cu NMI mediu maxim față de toate celelalte rulări (medoid partition), apoi verific că are și $Q$ competitiv.